# Comparación de tres formulaciones: dos particiones de CO₂
**Ensayo principal: CO₂ calibrado con LAB019–021 y holdout LAB016–018.**
**Ensayo inverso: CO₂ calibrado con LAB016–018 y holdout LAB019–021.**

Este notebook sólo carga parámetros: no optimiza. Las figuras de CO₂ se muestran separadas para ambas particiones; los títulos de cada panel indican su función. Las tres formulaciones se han ajustado al mismo objetivo de medianas horarias del sensor crudo, con igual ponderación por fermentación.

**La partición química es distinta y no cambia:** química calibrada en LAB004–009; LAB019–021 son holdout químico. En el ensayo principal LAB019–021 es calibración de CO₂ y, a la vez, holdout químico; no es un holdout conjunto del modelo completo.

**Figuras:** primero CO₂ emitido, después G+F de calibración química y holdout químico, estados químicos en holdout y diagnósticos de depósito/transferencia para cada ajuste de CO₂. En estos diagnósticos las curvas son predicciones internas antes de la ganancia de observación; sólo el punto inicial Cd es medido. En la comparación con el sensor la predicción incluye esa ganancia.

La solubilidad de los modelos nuevos sigue PROVISIONAL. Cd0 de LAB021 es provisional y en LAB016–018 se asume cero por falta de medición. El histórico conserva ecuaciones y theta_nv2; sus coeficientes de CO₂ se reajustan para cada partición sin alterar los ajustes históricos originales. No se elige un modelo sólo por el error de calibración.


In [ ]:
from pathlib import Path
import sys,json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/sources.csv').exists())
sys.dont_write_bytecode=True
sys.path.insert(0,str(ROOT/'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
assert Path(sys.prefix).name.lower()=='fermentation'


In [ ]:
from wine_fermentation.comparison import run_both,PRIMARY_TRAINING
from IPython.display import Markdown
experiments=run_both(ROOT,fit_gas=False)
result=experiments[PRIMARY_TRAINING]
combined=pd.concat([res['metrics'] for res in experiments.values()],ignore_index=True)
for key,res in experiments.items():
    display(Markdown('## CO₂ '+('principal' if key==PRIMARY_TRAINING else 'inverso')+': '+key))
    display(res['metrics'])
    display(pd.DataFrame(res['gas_fits']).T)
display(Markdown('## Media de RMSE por fermentación [g/(L·h)] · calibración y holdout separados'))
display(combined.groupby(['model','training','split']).RMSE.mean().unstack(['training','split']))
paired=combined.pivot(index=['model','lab'],columns='training',values='RMSE')
paired['delta_RMSE_principal_menos_inversa']=paired['train_019_021']-paired['train_016_018']
paired['rol_principal']=['Calibración CO₂' if lab in result['training_labs'] else 'Holdout CO₂' for _,lab in paired.index]
display(Markdown('## Mismo LAB, dos ajustes · delta negativo = menor RMSE con el ajuste principal; el rol del LAB cambia'))
display(paired)
display(Markdown('## Holdout químico LAB019–021 · parámetros químicos idénticos en ambos ensayos de CO₂'))
display(result['chemical_metrics'])


In [ ]:
colors={'historical':'#0072B2','without_transporters':'#D55E00','with_transporters':'#009E73'}
names={'historical':'Histórico · CO₂ reajustado','without_transporters':'Sin transportadores','with_transporters':'Con transportadores'}
for key,res in experiments.items():
    ordered_labs=res['training_labs']+res['holdout_labs']
    train_label=', '.join(res['training_labs']);holdout_label=', '.join(res['holdout_labs'])
    fig,axes=plt.subplots(2,3,figsize=(15,9))
    fig.subplots_adjust(top=.77,bottom=.08,hspace=.43,wspace=.26)
    fig.suptitle('CO₂ emitido'+'\nCalibración: '+train_label+' | Holdout: '+holdout_label,y=.99,fontsize=14)
    for ax,lab in zip(axes.flat,ordered_labs):
        obs=res['data'].query('lab==@lab and time_h>=0 and record_kind=="native"')
        ax.plot(obs.time_h,obs.co2_raw_g_l_h,color='gray',lw=.4,alpha=.4,label='Sensor crudo')
        for name in colors:
            f=res['curves'][name,lab];ax.plot(f.index,f.qpred,color=colors[name],label=names[name])
        split='Calibración CO₂' if lab in res['training_labs'] else 'Holdout CO₂ · sin reajuste'
        ax.set_title(lab+' · '+split,fontsize=11)
        ax.set_ylim(0,1.2)
        ax.set_xlabel('Tiempo desde inoculación [h]');ax.set_ylabel('CO₂ emitido [g/(L·h)]');ax.grid(alpha=.25)
    handles,labels=axes[0,0].get_legend_handles_labels()
    fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(.5,.90),ncol=4,title='Datos y formulaciones · química fija, coeficientes CO₂ ajustados sólo con la fila superior')
    plt.show()

chemical_names={**names,'historical':'Histórico · theta_nv2 fijo'}


## Glucosa + fructosa
La suma experimental requiere ambos azúcares en la misma muestra; no se imputan datos faltantes. Primero se muestran los ensayos usados en el ajuste químico de los modelos nuevos y después el holdout, sin recalibrar.

In [ ]:
from wine_fermentation.data import natural_batches
from wine_fermentation.shared.simulation import simulate
natural_observations,natural_training_batches,_=natural_batches(ROOT)
chemical_parameters={'historical':json.loads((ROOT/'configs/theta/theta_nv2.json').read_text(encoding='utf-8'))}
for name in ('without_transporters','with_transporters'):
    chemical_parameters[name]=json.loads((ROOT/f'configs/models/{name}/fit.json').read_text(encoding='utf-8'))['parameters']
fig,axes=plt.subplots(2,3,figsize=(15,9))
fig.subplots_adjust(top=.79,bottom=.08,hspace=.43,wspace=.26)
fig.suptitle('Glucosa + fructosa · mosto natural LAB004–009\nDatos de calibración química de los modelos nuevos · histórico congelado',y=.99,fontsize=15)
for ax,batch in zip(axes.flat,natural_training_batches):
    obs=natural_observations.loc[natural_observations.batch.eq(batch.batch)].dropna(subset=['G_g_l','F_g_l'])
    obs=obs.loc[obs.time_h.gt(0)]
    ax.scatter(obs.time_h,obs.G_g_l+obs.F_g_l,color='black',s=16,label='Mediciones G + F (misma muestra)')
    ax.scatter([0],[batch.initials['G']+batch.initials['F']],facecolors='none',edgecolors='black',s=28,label='Condición inicial')
    for name in colors:
        f=simulate(name,batch,chemical_parameters[name])
        ax.plot(f.index,f.G+f.F,color=colors[name],label=chemical_names[name])
    ax.set_title(batch.batch+' · Calibración química',fontsize=11)
    ax.set_xlabel('Tiempo desde inoculación [h]');ax.set_ylabel('Glucosa + fructosa [g/L]');ax.grid(alpha=.25)
handles,labels=axes[0,0].get_legend_handles_labels()
fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(.5,.90),ncol=3,title='Datos y formulaciones')
plt.show()


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4.8))
fig.subplots_adjust(top=.60,bottom=.15,wspace=.28)
fig.suptitle('Glucosa + fructosa · predicción química en holdout LAB019–021\nParámetros congelados · sin reajuste a estos ensayos',y=.99,fontsize=14)
for ax,lab in zip(axes,('LAB019','LAB020','LAB021')):
    obs=result['V'].chem.query('lab==@lab and is_NUT==False').dropna(subset=['G_g_l','F_g_l'])
    observed=obs.loc[obs.t_h.gt(0)]
    initial=obs.loc[obs.t_h.le(0)]
    ax.scatter(observed.t_h,observed.G_g_l+observed.F_g_l,color='black',s=16,label='Mediciones G + F (misma muestra)')
    ax.scatter(initial.t_h,initial.G_g_l+initial.F_g_l,facecolors='none',edgecolors='black',s=28,label='Muestra inicial')
    for name in colors:
        f=result['chemical'][name,lab]
        ax.plot(f.index,f.G+f.F,color=colors[name],label=chemical_names[name])
    ax.set_title(lab+' · Holdout',fontsize=11)
    ax.set_xlabel('Tiempo desde inoculación [h]');ax.set_ylabel('Glucosa + fructosa [g/L]');ax.grid(alpha=.25)
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(.5,.85),ncol=3,title='Datos y formulaciones')
plt.show()


In [ ]:
columns={'X':'X_viable_g_l','N':'N_kg_m3','G':'G_g_l','F':'F_g_l','E':'E_g_l'}
state_names={'X':'Biomasa viable','N':'YAN','G':'Glucosa','F':'Fructosa','E':'Etanol'}
for state,col in columns.items():
    fig,axes=plt.subplots(1,3,figsize=(15,4.8))
    fig.subplots_adjust(top=.68,bottom=.15,wspace=.28)
    fig.suptitle(state_names[state]+' · predicción química en holdout LAB019–021\nParámetros congelados · sin reajuste a estos ensayos',y=.99,fontsize=14)
    for ax,lab in zip(axes,('LAB019','LAB020','LAB021')):
        obs=result['V'].chem.query('lab==@lab and is_NUT==False').dropna(subset=[col])
        ax.scatter(obs.t_h,obs[col],color='black',s=14,label='Mediciones offline')
        for name in colors:
            f=result['chemical'][name,lab];ax.plot(f.index,f[state],color=colors[name],label=chemical_names[name])
        ax.set_title(lab+' · Holdout',fontsize=11)
        ax.set_xlabel('Tiempo desde inoculación [h]');ax.set_ylabel(state_names[state]+' [g/L]');ax.grid(alpha=.25)
    handles,labels=axes[0].get_legend_handles_labels()
    fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(.5,.85),ncol=4,title='Datos y formulaciones')
    plt.show()


In [ ]:
for key,res in experiments.items():
    fig,axes=plt.subplots(2,3,figsize=(16,10))
    fig.subplots_adjust(top=.66,bottom=.08,hspace=.44,wspace=.27)
    fig.suptitle('CO₂ disuelto y transferencia · ajuste '+('PRINCIPAL' if key==PRIMARY_TRAINING else 'INVERSO')+'\nLAB019–021: '+('calibración CO₂' if key==PRIMARY_TRAINING else 'holdout CO₂')+' | Solubilidad PROVISIONAL | Antes de ganancia',y=.99,fontsize=14)
    for j,lab in enumerate(('LAB019','LAB020','LAB021')):
        for name in ('without_transporters','with_transporters'):
            f=res['curves'][name,lab]
            axes[0,j].plot(f.index,f.Cd,color=colors[name],label=names[name]+' · Cd')
            axes[0,j].plot(f.index,f.Csat,color=colors[name],ls=':',label=names[name]+' · Csat provisional')
            axes[1,j].plot(f.index,f.qprod,color=colors[name],ls='--',label=names[name]+' · producción')
            axes[1,j].plot(f.index,f.qgas,color=colors[name],label=names[name]+' · emisión')
        axes[0,j].scatter([0],[{'LAB019':.907,'LAB020':.730,'LAB021':.829}[lab]],color='black',s=22,label='Cd inicial medido (021 provisional)')
        axes[0,j].set_title(lab+' · '+('Calibración CO₂' if lab in res['training_labs'] else 'Holdout CO₂')+' · Depósito',fontsize=11)
        axes[1,j].set_title(lab+' · Producción y emisión (predicción)',fontsize=11)
        for ax in axes[:,j]:ax.set_xlabel('Tiempo desde inoculación [h]');ax.grid(alpha=.25)
        axes[0,j].set_ylabel('CO₂ disuelto [g/L]')
        axes[1,j].set_ylabel('Caudal de CO₂ [g/(L·h)]')
    handles,labels=axes[0,0].get_legend_handles_labels()
    h2,l2=axes[1,0].get_legend_handles_labels()
    fig.legend(handles+h2,labels+l2,loc='upper center',bbox_to_anchor=(.5,.91),ncol=3,title='Curvas calculadas: Cd = concentración disuelta; Csat = saturación | Punto: condición inicial medida',fontsize=9,title_fontsize=10)
    plt.show()
